In [3]:
import pandas as pd
import numpy as np
import os

print("=" *60)
print("DATA VALIDATION AND CLEANING")
print("=" *60)


#1.loading datasets and getting baseline counts
prices_path = "data/raw/prices/raw_maize_market_prices.csv"
weather_path = "data/raw/weather/raw_county_weather.csv"

df_prices = pd.read_csv(prices_path)
df_weather = pd.read_csv(weather_path)

rows_received_prices = len(df_prices)
rows_received_weather = len(df_weather)

#2.view the 5 rows to see the columns and values look like
print("\n---First 5 Rows of Raw Prices Data ---")
display(df_prices.head())

#3.view data types and  non null counts (to see where blanks exist)
print("\n---Data Information and Null counts---")
df_prices.info()
#4.check the exact missing values dynamically
print("\n---Exact Missing Values per column ---")
print(df_prices.isna().sum())
#5.check unique values in categorical columns to see inconsistencies(eg messycounties or units)
print("\n---Unique Units in Raw Data (Spotting inconsistencies)---")
print(df_prices['unit'].unique())

#PART 2:CLEANING
print("=" * 60)
print("RUNNING DATA CLEANING")
print("=" * 60)

rows_received_prices = len(df_prices)
rows_received_weather = len(df_weather)

#1.track initial missing values dynamically from our inspection
initial_missing_prices = df_prices['wholesale_price'].isna().sum()
initial_missing_retail = df_prices['retail_price_per_kg'].isna().sum()
initial_missing_supply = df_prices['supply_volume_tonnes'].isna().sum()

#2.handling duplicate records
duplicates_count = df_prices.duplicated().sum()
df_prices_dedup = df_prices.drop_duplicates().copy()

#3.cleaning county and market names
df_prices_dedup['county_name'] = df_prices_dedup['county_name'].astype(str).str.strip().str.title()
df_prices_dedup['market_name'] = df_prices_dedup['market_name'].astype(str).str.strip().str.title()

#4.cleaning currency formats and malformed numbers
def clean_currency_to_float(val):
    if pd.isna(val):
        return np.nan

    if isinstance(val,(int,float)):
        return float(val)
    cleaned = str(val).upper().replace("KES","").replace(",","").strip()
    try:
        return float(cleaned)
    except ValueError:
        return np.nan
    
df_prices_dedup['wholesale_price_cleaned'] = df_prices_dedup['wholesale_price'].apply(clean_currency_to_float)
malformed_prices_count = df_prices_dedup['wholesale_price_cleaned'].isna().sum()

#5.standardizing measurements units

def standardize_unit(unit_str):
     if pd.isna(unit_str):
         return "unknown"
     u = str(unit_str).lower().strip()
     if "90" in u:
         return "90kg bag"
     return u
df_prices_dedup['standardized_unit']=df_prices_dedup['unit'].apply(standardize_unit)

#6.validating dates
df_prices_dedup['reporting_date'] = pd.to_datetime(df_prices_dedup['reporting_date'],errors='coerce')
invalid_date_count = df_prices_dedup['reporting_date'].isna().sum()
df_prices_clean_dates = df_prices_dedup.dropna(subset=['reporting_date']).copy()

#7.outlier detection and impossible values
outlier_mask = (df_prices_clean_dates['wholesale_price_cleaned'] < 1500) | (df_prices_clean_dates['wholesale_price_cleaned']>10000)
outliers_count = outlier_mask.sum()
df_valid_prices = df_prices_clean_dates[~outlier_mask].dropna(subset=['wholesale_price_cleaned']).copy()

#8.saving clean data and generating audit report
min_date = df_valid_prices['reporting_date'].min().strftime('%Y-%m-%d')
max_date = df_valid_prices['reporting_date'].max().strftime('%Y-%m-%d')
unique_counties = df_valid_prices['county_name'].nunique()
unique_markets = df_valid_prices['market_name'].nunique()

#9.saving cleaned dataset
os.makedirs("data/clean", exist_ok=True)
clean_prices_output = "data/clean/clean_maize_market_prices.csv"
df_valid_prices.to_csv(clean_prices_output, index=False)

# 7. Automated Data-Quality Report
valid_records_count = len(df_valid_prices)
total_removed = rows_received_prices - valid_records_count

print("\n" + "=" * 60)
print(" 📊 AUTOMATED DATA-QUALITY REPORT (PHASE 2 - STEP 6)")
print("=" * 60)
print(f" Raw Records Received:        {rows_received_prices:,}")
print(f" Valid Clean Records:         {valid_records_count:,}")
print(f" Total Records Removed/Fixed: {total_removed:,}")
print("-" * 60)
print(" 🔍 ANOMALY & CLEANING BREAKDOWN:")
print(f"   • Duplicate Bulletins Removed:   {duplicates_count:,} rows")
print(f"   • Malformed/Unparsable Prices:   {malformed_prices_count:,} rows")
print(f"   • Invalid Date Formats Dropped:  {invalid_date_count:,} rows")
print(f"   • Statistical Outliers Flagged:  {outliers_count:,} rows")
print("-" * 60)
print(" 📉 MISSING-VALUE PERCENTAGES:")
print(f"   • Wholesale Price Missing:       {(initial_missing_prices / rows_received_prices) * 100:.2f}%")
print(f"   • Retail Price Missing:          {(initial_missing_retail / rows_received_prices) * 100:.2f}%")
print(f"   • Supply Volume Missing:         {(initial_missing_supply / rows_received_prices) * 100:.2f}%")
print("-" * 60)
print(" 🌍 COVERAGE METRICS:")
print(f"   • Date Coverage:                 {min_date} to {max_date}")
print(f"   • Geographic Coverage:           {unique_counties} Counties | {unique_markets} Markets")
print("=" * 60)
print(f"✅ Success! Clean validated data saved to: {clean_prices_output}")



DATA VALIDATION AND CLEANING

---First 5 Rows of Raw Prices Data ---


,reporting_date,county_name,market_name,commodity,wholesale_price,unit,retail_price_per_kg,supply_volume_tonnes
0,2024-01-01,Machakos,Machakos Town,Dry Maize,4926,90kg bag,64.36,30.0
1,2024-01-01,Machakos,Athi River,Dry Maize,"KES 3,930",90-kg bag,54.32,52.6
2,2024-01-01,Kitui,Kitui Central,Dry Maize,"KES 3,821",90kg bag,NaN,96.6
3,2024-01-01,Makueni,Wote,Dry Maize,5390,90kg bag,70.68,NaN
4,2024-01-01,Nairobi,Wakulima,Dry Maize,4400,Bag (90kg),54.42,14.8



---Data Information and Null counts---
<class 'pandas.DataFrame'>
RangeIndex: 1130 entries, 0 to 1129
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   reporting_date        1130 non-null   str    
 1   county_name           1130 non-null   str    
 2   market_name           1130 non-null   str    
 3   commodity             1130 non-null   str    
 4   wholesale_price       1130 non-null   str    
 5   unit                  1130 non-null   str    
 6   retail_price_per_kg   1077 non-null   float64
 7   supply_volume_tonnes  1042 non-null   float64
dtypes: float64(2), str(6)
memory usage: 126.1 KB

---Exact Missing Values per column ---
reporting_date           0
county_name              0
market_name              0
commodity                0
wholesale_price          0
unit                     0
retail_price_per_kg     53
supply_volume_tonnes    88
dtype: int64

---Unique Units in Raw Data (Spotti